# 0) Import the Dataset & Setup

In [8]:
BASE_DATASET_FOLDER = "C:/Users/User/Documents/IIT Stage 2/IIT Stage 2 Semester 1/CM2603  Data Science Group Project/ECG_DATA/train"   # folder with 4 category subfolders
OUTPUT_FOLDER = "C:/Users/User/Documents/IIT Stage 2/IIT Stage 2 Semester 1/CM2603  Data Science Group Project/ECG_PREPROCESS_OUTPUT"  # where CSVs (and optional images) will be saved

SAVE_INTERMEDIATE_IMAGES = False


In [9]:
import os
os.makedirs(OUTPUT_FOLDER, exist_ok=True)
print("Outputs will be saved to:", OUTPUT_FOLDER)


Outputs will be saved to: C:/Users/User/Documents/IIT Stage 2/IIT Stage 2 Semester 1/CM2603  Data Science Group Project/ECG_PREPROCESS_OUTPUT2


# 1) Utilities & Preprocessing functions

In [10]:
# All core functions in one cell.
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import glob

# ---------- STEP A: image preprocessing (grayscale, denoise, threshold) ----------
def preprocess_step1_image(img):
    """Take BGR image (cv2.imread) and return preprocessed binary image (threshold)."""
    if img is None:
        return None
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    denoised = cv2.medianBlur(gray, 3)
    thr = cv2.adaptiveThreshold(
        denoised, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV,
        35, 7
    )
    return thr

# ---------- STEP B: smart crop into 12 leads ----------
def crop_12_leads_from_gray(gray_img):
    """Input: gray image (2D). Output: dict Lead_1..Lead_12 -> crop images (grayscale)."""
    h, w = gray_img.shape[:2]
    top_margin = int(0.18 * h)      # skip header; tweak if layout differs
    lead_height = int((h - top_margin) / 4)
    lead_width  = int(w / 3)

    leads = {}
    idx = 1
    for row in range(4):
        for col in range(3):
            y1 = top_margin + row * lead_height
            y2 = top_margin + (row + 1) * lead_height
            x1 = col * lead_width
            x2 = (col + 1) * lead_width
            crop = gray_img[y1:y2, x1:x2].copy()
            leads[f"Lead_{idx}"] = crop
            idx += 1
    return leads

# ---------- STEP C: clean lead (blur, threshold, morphology) ----------
def clean_lead_for_signal(lead_img):
    """Return single cleaned binary image for signal extraction (uint8)."""
    gray = lead_img if len(lead_img.shape) == 2 else cv2.cvtColor(lead_img, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5,5), 0)
    binary = cv2.adaptiveThreshold(blur, 255,
                                   cv2.ADAPTIVE_THRESH_MEAN_C,
                                   cv2.THRESH_BINARY_INV,
                                   41, 5)
    kernel = np.ones((3,3), np.uint8)
    clean = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)
    return clean

# ---------- STEP D: extract 1D signal from cleaned lead ----------
def extract_signal(clean_img):
    """Return normalized 1D signal (length = width of clean_img). Values in [0,1]."""
    clean = clean_img.astype(np.uint8)
    h, w = clean.shape
    signal = []
    for x in range(w):
        column = clean[:, x]
        pts = np.where(column > 0)[0]
        if len(pts) == 0:
            signal.append(signal[-1] if len(signal) > 0 else h//2)
        else:
            y = int(np.mean(pts))
            signal.append(y)
    sig = np.array(signal, dtype=float)
    # normalize 0..1; handle flat signal
    if sig.max() - sig.min() < 1e-8:
        return np.zeros_like(sig)
    return (sig - sig.min()) / (sig.max() - sig.min())

# ---------- STEP E: resample to fixed length ----------
def resample_signal(sig, target_len):
    """Resample 1D vector sig to length target_len via linear interpolation."""
    if len(sig) == target_len:
        return sig
    if len(sig) == 0:
        return np.zeros(target_len, dtype=float)
    old_x = np.linspace(0, 1, num=len(sig))
    new_x = np.linspace(0, 1, num=target_len)
    return np.interp(new_x, old_x, sig)

# ---------- small helper to write images if requested ----------
def save_intermediate_images(image_dict, out_dir: Path):
    out_dir.mkdir(parents=True, exist_ok=True)
    for name, img in image_dict.items():
        # write as png
        cv2.imwrite(str(out_dir / f"{name}.png"), img)


# 2) Automatically compute TARGET_LEAD_LENGTH from dataset

In [11]:
from tqdm import tqdm

def compute_target_lead_length(dataset_root):
    """
    Compute median width of ECG leads across the dataset.
    This ensures consistent and data-driven resampling length.
    """
    lead_widths = []

    image_paths = (
        list(Path(dataset_root).rglob("*.jpg")) +
        list(Path(dataset_root).rglob("*.png"))
    )

    print(f"[INFO] Found {len(image_paths)} ECG images for lead length estimation")

    for img_path in tqdm(image_paths):
        img = cv2.imread(str(img_path))
        if img is None:
            continue

        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        leads = crop_12_leads_from_gray(gray)

        for lead_img in leads.values():
            h, w = lead_img.shape
            lead_widths.append(w)

    lead_widths = np.array(lead_widths)

    print(
        f"[INFO] Lead width stats → "
        f"min={lead_widths.min()}, "
        f"median={int(np.median(lead_widths))}, "
        f"max={lead_widths.max()}"
    )

    return int(np.median(lead_widths))

In [12]:
TARGET_LEAD_LENGTH = compute_target_lead_length(BASE_DATASET_FOLDER)
print("Using TARGET_LEAD_LENGTH =", TARGET_LEAD_LENGTH)


[INFO] Found 3023 ECG images for lead length estimation


100%|██████████| 3023/3023 [01:05<00:00, 46.33it/s]

[INFO] Lead width stats → min=737, median=737, max=737
Using TARGET_LEAD_LENGTH = 737


# 3) Single-image pipeline function

In [13]:
def process_single_ecg_image(img_path, target_len=TARGET_LEAD_LENGTH, save_images=False, save_folder=None):
    """
    Process one ECG image file path.
    Returns: flattened vector (1D numpy) length = 12 * target_len
    If save_images True and save_folder provided, saves intermediate images there.
    """
    p = Path(img_path)
    bgr = cv2.imread(str(p))
    

    
    if bgr is None:
        print("WARN: cannot read", img_path)
        return None

    # step1 preprocess
    thr = preprocess_step1_image(bgr)            # binary image
   
    
    # optionally save step1
    if save_images and save_folder:
        save_intermediate_images({"step1_threshold": thr}, Path(save_folder) / p.stem)

    # For cropping we need grayscale original (not binary) to keep waveform shapes
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    
    # step2 crop 12 leads
    leads = crop_12_leads_from_gray(gray)
    # optional save crops
    if save_images and save_folder:
        save_intermediate_images({f"crop_{k}": v for k,v in leads.items()}, Path(save_folder) / p.stem / "crops")

    # step3-clean & extract & resample
    all_lead_signals = []
    for i in range(1, 13):
        lead = leads[f"Lead_{i}"]
        clean = clean_lead_for_signal(lead)

        sig = extract_signal(clean)
        sig_rs = resample_signal(sig, target_len)
        all_lead_signals.append(sig_rs)
        # optionally save cleaned lead images
        if save_images and save_folder:
            save_intermediate_images({f"clean_lead_{i}": clean}, Path(save_folder) / p.stem / "cleaned")

    # flatten ordering: Lead1_all, Lead2_all, ..., Lead12_all
    flattened = np.concatenate(all_lead_signals, axis=0)
    return flattened


In [14]:
# Example test: pick one file from a category
sample_file = list(Path(BASE_DATASET_FOLDER).glob("*/*.jpg"))[:1]
if sample_file:
    sample_file = str(sample_file[0])
    vec = process_single_ecg_image(sample_file, target_len=TARGET_LEAD_LENGTH, save_images=False)
    print("vector length:", None if vec is None else vec.shape)
else:
    print("No sample files found - check BASE_DATASET_FOLDER")


vector length: (8844,)


# 4) Process an entire category folder and save CSV (one row per image)

In [15]:
def process_category_folder(category_folder, output_csv_path, target_len=TARGET_LEAD_LENGTH,
                            save_images=False, intermediate_save_base=None):
    """
    Process all images in category_folder and save flattened CSV with one row per image.
    Returns a DataFrame (rows = images, cols = 12*target_len).
    """
    category_folder = Path(category_folder)
    image_paths = sorted([*category_folder.rglob("*.jpg"),
                          *category_folder.rglob("*.jpeg"),
                          *category_folder.rglob("*.png")])

    rows = []
    filenames = []
    print(f"[INFO] Found {len(image_paths)} images in {category_folder.name}")

    for img_path in image_paths:
        flattened = process_single_ecg_image(str(img_path), target_len=target_len,
                                             save_images=save_images,
                                             save_folder=(Path(intermediate_save_base) if intermediate_save_base else None))
        if flattened is None:
            continue
        rows.append(flattened)
        filenames.append(img_path.name)

    if len(rows) == 0:
        print("[WARN] No processed rows for", category_folder)
        return None

    arr = np.vstack(rows)  # shape = (n_images, 12*target_len)

    # build column names
    cols = []
    for lead in range(1, 13):
        for t in range(target_len):
            cols.append(f"Lead{lead}_{t}")

    df = pd.DataFrame(arr, columns=cols)
    # optional: add filename column
    df.insert(0, "filename", filenames)
    df.to_csv(output_csv_path, index=False)
    print("[OK] Saved category CSV:", output_csv_path, "shape:", df.shape)
    return df


In [16]:
# Example for one category folder
cat = list(Path(BASE_DATASET_FOLDER).iterdir())[0]  # first category folder
out_csv = Path(OUTPUT_FOLDER) / f"{cat.name}_flattened.csv"
df_test = process_category_folder(cat, out_csv, target_len=TARGET_LEAD_LENGTH,
                                  save_images=SAVE_INTERMEDIATE_IMAGES,
                                  intermediate_save_base=Path(OUTPUT_FOLDER) / "intermediates")


[INFO] Found 956 images in ECG Images of Myocardial Infarction Patients (240x12=2880)
[OK] Saved category CSV: C:\Users\User\Documents\IIT Stage 2\IIT Stage 2 Semester 1\CM2603  Data Science Group Project\ECG_PREPROCESS_OUTPUT2\ECG Images of Myocardial Infarction Patients (240x12=2880)_flattened.csv shape: (956, 8845)


# 5) Batch-run for all category folders

In [17]:
all_folders = [p for p in Path(BASE_DATASET_FOLDER).iterdir() if p.is_dir()]
print("Categories to process:", [p.name for p in all_folders])

for cat in all_folders:
    out_csv = Path(OUTPUT_FOLDER) / f"{cat.name}_flattened.csv"
    # skip if already exists (optional)
    if out_csv.exists():
        print("[SKIP] Already exists:", out_csv)
        continue
    process_category_folder(cat, out_csv, target_len=TARGET_LEAD_LENGTH,
                            save_images=SAVE_INTERMEDIATE_IMAGES,
                            intermediate_save_base=Path(OUTPUT_FOLDER) / "intermediates")


Categories to process: ['ECG Images of Myocardial Infarction Patients (240x12=2880)', 'ECG Images of Patient that have abnormal heartbeat (233x12=2796)', 'ECG Images of Patient that have History of MI (172x12=2064)', 'Normal Person ECG Images (284x12=3408)']
[SKIP] Already exists: C:\Users\User\Documents\IIT Stage 2\IIT Stage 2 Semester 1\CM2603  Data Science Group Project\ECG_PREPROCESS_OUTPUT2\ECG Images of Myocardial Infarction Patients (240x12=2880)_flattened.csv
[INFO] Found 699 images in ECG Images of Patient that have abnormal heartbeat (233x12=2796)
[OK] Saved category CSV: C:\Users\User\Documents\IIT Stage 2\IIT Stage 2 Semester 1\CM2603  Data Science Group Project\ECG_PREPROCESS_OUTPUT2\ECG Images of Patient that have abnormal heartbeat (233x12=2796)_flattened.csv shape: (699, 8845)
[INFO] Found 516 images in ECG Images of Patient that have History of MI (172x12=2064)
[OK] Saved category CSV: C:\Users\User\Documents\IIT Stage 2\IIT Stage 2 Semester 1\CM2603  Data Science Grou

# 6) Quick sanity checks

In [19]:
total_ecgs = 0

for csv_file in Path(OUTPUT_FOLDER).glob("*_flattened.csv"):
    df = pd.read_csv(csv_file)
    print(csv_file.name, "->", df.shape[0])
    total_ecgs += df.shape[0]

print("TOTAL preprocessed ECG images:", total_ecgs)

ECG Images of Myocardial Infarction Patients (240x12=2880)_flattened.csv -> 956
ECG Images of Patient that have abnormal heartbeat (233x12=2796)_flattened.csv -> 699
ECG Images of Patient that have History of MI (172x12=2064)_flattened.csv -> 516
Normal Person ECG Images (284x12=3408)_flattened.csv -> 852
TOTAL preprocessed ECG images: 3023
